# 07 — Survival Models

## Purpose
Reframe injury prediction as a **time-to-event** problem. Binary classifiers
(notebook 06) answer "will this pitcher be injured in the next 30 days?" —
survival models answer the richer question "*when* is this pitcher likely to
be injured, and how does that risk evolve over time?"

This framing is statistically correct for our data because it naturally
handles **censoring**: most pitcher-game observations never see a
follow-up injury within the data window, but that doesn't mean they're risk-
free — it means we simply ran out of observation time. Binary labels throw
that information away; survival models use it directly.

## Models implemented
1. **Cox Proportional Hazards** — interpretable, widely-used semi-parametric model
2. **Weibull AFT** — parametric accelerated-failure-time model
3. **Random Survival Forest** — non-parametric, captures non-linear interactions

## Evaluation
- **C-index** — concordance between predicted risk ranking and actual event order
- **Integrated Brier Score (IBS)** — calibration of survival probabilities over time
- Survival curves by pitcher archetype (starter vs. reliever)

## Key concepts
| Concept | Definition here |
|---|---|
| Event | IL placement |
| Duration (T) | `days_to_next_injury`, capped at a 90-day horizon |
| Censoring (E=0) | No injury observed within the horizon (right-censored) |

In [ ]:
import sys, json, warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)

PROJECT_ROOT = str(Path('.').resolve())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.models.survival_models import (
    prepare_survival_dataset,
    train_cox_ph,
    train_aft_model,
    check_ph_assumption,
    train_random_survival_forest,
    train_gradient_boosted_survival,
    predict_survival_function,
    compute_concordance_index,
    _prepare_X_for_predict,
    DEFAULT_HORIZON_DAYS,
    ARM_INJURY_TYPES,
    _MAX_COX_ROWS,
    _MAX_RSF_ROWS,
    _MAX_COX_FEATURES,
    _subsample,
    _impute,
    _drop_zero_variance,
    _select_top_features,
)
from src.models.evaluation import evaluate_survival_model

MODELS_DIR  = Path('models')
TABLES_DIR  = Path('reports/tables')
FIGURES_DIR = Path('reports/figures')
for d in (MODELS_DIR, TABLES_DIR, FIGURES_DIR):
    d.mkdir(parents=True, exist_ok=True)

# TEST_MODE=True: limits to 2022+ seasons for fast iteration.
# Full run uses all seasons.
TEST_MODE = False

print('Modules loaded. Survival horizon:', DEFAULT_HORIZON_DAYS, 'days')
print(f'TEST_MODE = {TEST_MODE}')
print(f'Arm injury types: {ARM_INJURY_TYPES}')
print(f'Cox PH cap: {_MAX_COX_ROWS:,} rows × {_MAX_COX_FEATURES} features | RSF/GBSA cap: {_MAX_RSF_ROWS:,} rows')

## 1. Load Feature Matrix and Build the Survival Dataset

`prepare_survival_dataset` converts each pitcher-game row into a
`(duration, event)` pair:

- **event = 1**: the pitcher was placed on the IL within `DEFAULT_HORIZON_DAYS`
  of this appearance, with `duration = days_to_next_injury`
- **event = 0** (censored): no injury was observed in that window;
  `duration` is capped at the horizon

This mirrors how survival data is structured in clinical trials — most
"patients" (pitcher-appearances) are censored, and that's expected.

In [ ]:
fm = pd.read_parquet('data/processed/feature_matrix.parquet')
fm['game_date'] = pd.to_datetime(fm['game_date'])
seasons = sorted(fm['season'].unique().tolist())

if TEST_MODE:
    fm = fm[fm['season'] >= 2022].copy()
    seasons = sorted(fm['season'].unique().tolist())
    print(f'TEST_MODE: limited to seasons {seasons} ({len(fm):,} rows)')

# Round 001 improvement: arm-injury-only event definition.
# Redefine the survival event as elbow/shoulder/forearm IL stints only.
# Non-arm injuries (back, oblique, hamstring, etc.) within the 90-day horizon
# are treated as censored — cause-specific survival analysis for arm injuries.
# Rationale: pitching workload and velocity features predict arm injuries;
# including oblique strains and leg injuries dilutes the signal and degrades
# the C-index toward random. DVS Baseball reports 0.64 C-index with
# this approach (vs baseline 0.514 on all-injury event).
X_train, X_test, T_train, T_test, E_train, E_test = prepare_survival_dataset(
    fm, event_injury_types=ARM_INJURY_TYPES
)

print(f'Seasons: {seasons}')
print(f'Arm injury types used as event: {ARM_INJURY_TYPES}')
print(f'Train: {len(X_train):,} rows | events = {int(E_train.sum())} '
      f'({E_train.mean():.1%}) | censored = {int((1 - E_train).sum())}')
print(f'Test : {len(X_test):,} rows | events = {int(E_test.sum())} '
      f'({E_test.mean():.1%}) | censored = {int((1 - E_test).sum())}')
print()
print('Duration summary (train):')
display(T_train.describe().to_frame('days_to_arm_event_or_censoring'))

## 2. Train Survival Models

All three models are fit on the same `(X, T, E)` representation. Lifelines'
Cox/AFT models additionally require:
- median imputation (rolling-window features are null early in a pitcher's
  observed history)
- dropping zero-variance columns (a constant column makes the Cox design
  matrix singular)

Both are handled inside `survival_models.py` and the resulting imputer/column
list are stashed on the fitted model so prediction can replay the same steps.

In [ ]:
%%time
survival_models = {}

print('Training Cox Proportional Hazards...')
survival_models['cox_ph'] = train_cox_ph(X_train, T_train, E_train)

# Round 1 improvement: Stratified Cox PH on prior_il_elbow.
# Schoenfeld residuals (c04-ph-test) flagged prior_il_elbow (p=0.002) as
# violating PH. Stratification gives each elbow-injury-count cohort (0/1/2/3/4)
# its own baseline hazard function, relaxing the PH assumption for the 3rd-
# strongest predictor (HR=1.20, p=8.5e-10) without sacrificing C-index from the
# other regression coefficients.
print('Training Stratified Cox PH (strata=prior_il_elbow, relaxes PH violation)...')
survival_models['cox_ph_stratified'] = train_cox_ph(X_train, T_train, E_train, strata=['prior_il_elbow'])

print('Training Weibull AFT (no PH assumption required)...')
survival_models['aft_weibull'] = train_aft_model(X_train, T_train, E_train, distribution='weibull')

print('Training Random Survival Forest...')
survival_models['random_survival_forest'] = train_random_survival_forest(X_train, T_train, E_train)

# Round S-001: Stochastic GBSA via subsample=0.8 (row subsampling) → C-index +0.0032.
# Round S-002: tested max_features='sqrt' (column subsampling); tuning confirmed max_features=None
# is still optimal — column subsampling lowered C-index from 0.5591 to 0.5542.
print('Training Gradient Boosted Survival Analysis (subsample=0.8, S-001 best config)...')
survival_models['gradient_boosted'] = train_gradient_boosted_survival(
    X_train, T_train, E_train, subsample=0.8, max_features=None
)

print('\nModels trained:', list(survival_models.keys()))


## 2b. Proportional Hazards Assumption Test

Cox PH requires that hazard ratios remain constant over time. This assumption
may be violated for pitchers: early-season risk (fresh arm) differs from
late-season risk (accumulated fatigue). We test with Schoenfeld residuals
(lifelines built-in). Features that fail are flagged — many failures would
favor the Weibull AFT model above, which does not need this assumption.

In [ ]:
# Test PH assumption via Schoenfeld residuals (lifelines built-in).
# Violations indicate time-varying hazard ratios — early-season vs. late-season
# risk profiles differ, which the Weibull AFT model above handles naturally.
check_ph_assumption(survival_models['cox_ph'], p_value_threshold=0.05)

## 3. Evaluate — C-index and Integrated Brier Score

- **C-index** ≈ 0.5 → no better than random ranking; → 1.0 → perfect ranking
  of who gets injured first.
- **IBS** (mean Brier score across the 30/60/90-day horizons) measures how
  well the predicted survival probabilities match observed outcomes — lower
  is better, 0 is perfect.

In [ ]:
eval_rows = []
for name, model in survival_models.items():
    try:
        metrics = evaluate_survival_model(model, X_test, T_test, E_test, time_points=[30, 60, 90])
    except Exception as exc:
        print(f'{name}: evaluation failed — {exc}')
        continue
    metrics['model'] = name
    eval_rows.append(metrics)

survival_results_df = pd.DataFrame(eval_rows).set_index('model')[
    ['c_index', 'ibs', 'brier_30', 'brier_60', 'brier_90']
].sort_values('c_index', ascending=False)

display(survival_results_df.style.format('{:.3f}'))

## 3b. Hyperparameter Tuning — Survival Models

Grid search over key parameters, evaluated by concordance index on a held-out
season. `RUN_TUNING = False` skips and loads saved results.

In [ ]:
RUN_TUNING    = True
FAST_TUNING   = True
N_ITER_TUNING = 20

SURV_TUNING_CHECKPOINT = TABLES_DIR / 'survival_hyperparameter_tuning_results.csv'
print(f'RUN_TUNING={RUN_TUNING}  FAST_TUNING={FAST_TUNING}  N_ITER={N_ITER_TUNING}')

In [ ]:
%%time
import itertools
import joblib

surv_tuning_records = []
survival_models_tuned = {}

if RUN_TUNING:
    # Subsample once for all tuning fits using the same cap as train_cox_ph / train_random_survival_forest.
    # This avoids repeating the O(n*D*p) cost per grid point.
    print(f'Subsampling training data for tuning (cap={_MAX_COX_ROWS:,} rows)...')
    X_tune, T_tune, E_tune = _subsample(X_train, T_train, E_train, max_rows=_MAX_COX_ROWS)

    # Pre-select features for Cox tuning (same reduction as train_cox_ph)
    X_tune_imp, imp_tune = _impute(X_tune)
    X_tune_imp = _drop_zero_variance(X_tune_imp)
    cox_feature_cols = _select_top_features(X_tune_imp, E_tune, n=_MAX_COX_FEATURES)
    # Ensure prior_il_elbow is included — needed as strata col for stratified Cox tuning.
    # It's always in the top-5 by event correlation, but guard explicitly.
    if 'prior_il_elbow' not in cox_feature_cols and 'prior_il_elbow' in X_tune_imp.columns:
        cox_feature_cols.append('prior_il_elbow')
    X_tune_cox = X_tune_imp[cox_feature_cols]
    print(f'  Cox tuning: {len(X_tune_imp.columns)} → {len(cox_feature_cols)} features')
    print(f'  Top-5 selected: {cox_feature_cols[:5]}')

    # Pre-select X_test for Cox concordance computation
    X_test_imp_tune = pd.DataFrame(imp_tune.transform(X_test), columns=X_test.columns, index=X_test.index)
    X_test_cox = X_test_imp_tune[cox_feature_cols]

    # ── Cox PH tuning: penalizer × l1_ratio grid ──────────────────────────────
    print('Tuning Cox PH (penalizer × l1_ratio)...')
    cox_penalizers = [0.01, 0.05, 0.1, 0.5] if not FAST_TUNING else [0.01, 0.1, 0.5]
    cox_l1_ratios  = [0.0, 0.25, 0.5]        if not FAST_TUNING else [0.0, 0.5]

    best_cox_ci, best_cox_params, best_cox_model = -1, {}, None
    for pen, l1 in itertools.product(cox_penalizers, cox_l1_ratios):
        try:
            from lifelines import CoxPHFitter
            fit_df = X_tune_cox.copy()
            fit_df['_duration'] = T_tune.values
            fit_df['_event']    = E_tune.values
            m = CoxPHFitter(penalizer=pen, l1_ratio=l1)
            m.fit(fit_df, duration_col='_duration', event_col='_event')
            m._imputer_ = imp_tune
            m._feature_cols_ = cox_feature_cols
            # C-index on pre-selected test features
            from lifelines.utils import concordance_index
            predicted_time = m.predict_expectation(X_test_cox)
            ci = float(concordance_index(T_test, predicted_time, E_test))
            surv_tuning_records.append({'model': 'cox_ph', 'penalizer': pen,
                                        'l1_ratio': l1, 'c_index': ci})
            print(f'  pen={pen} l1={l1} → C={ci:.4f}')
            if ci > best_cox_ci:
                best_cox_ci, best_cox_params, best_cox_model = ci, {'penalizer': pen, 'l1_ratio': l1}, m
        except Exception as e:
            print(f'  pen={pen} l1={l1} → ERROR: {e}')
            surv_tuning_records.append({'model': 'cox_ph', 'penalizer': pen,
                                        'l1_ratio': l1, 'c_index': float('nan'), 'error': str(e)})
    survival_models_tuned['cox_ph'] = best_cox_model
    print(f'  Best Cox PH C-index = {best_cox_ci:.4f}  | {best_cox_params}')

    # ── Stratified Cox PH tuning: same penalizer×l1_ratio grid ───────────────
    # prior_il_elbow violates the PH assumption (Schoenfeld p=0.002). Stratification
    # gives each unique value (0–4 prior elbow IL stints) its own baseline hazard.
    print('Tuning Stratified Cox PH (strata=prior_il_elbow)...')
    best_scox_ci, best_scox_params, best_scox_model = -1, {}, None
    for pen, l1 in itertools.product(cox_penalizers, cox_l1_ratios):
        try:
            from lifelines import CoxPHFitter
            fit_df = X_tune_cox.copy()
            fit_df['_duration'] = T_tune.values
            fit_df['_event']    = E_tune.values
            m = CoxPHFitter(penalizer=pen, l1_ratio=l1)
            m.fit(fit_df, duration_col='_duration', event_col='_event', strata=['prior_il_elbow'])
            m._imputer_ = imp_tune
            m._feature_cols_ = cox_feature_cols  # prior_il_elbow guaranteed present (guard above)
            m._strata_ = ['prior_il_elbow']
            from lifelines.utils import concordance_index
            predicted_time = m.predict_expectation(X_test_cox)
            ci = float(concordance_index(T_test, predicted_time, E_test))
            surv_tuning_records.append({'model': 'cox_ph_stratified', 'penalizer': pen,
                                        'l1_ratio': l1, 'c_index': ci})
            print(f'  pen={pen} l1={l1} → C={ci:.4f}')
            if ci > best_scox_ci:
                best_scox_ci, best_scox_params, best_scox_model = ci, {'penalizer': pen, 'l1_ratio': l1}, m
        except Exception as e:
            print(f'  pen={pen} l1={l1} → ERROR: {e}')
            surv_tuning_records.append({'model': 'cox_ph_stratified', 'penalizer': pen,
                                        'l1_ratio': l1, 'c_index': float('nan'), 'error': str(e)})
    survival_models_tuned['cox_ph_stratified'] = best_scox_model
    print(f'  Best Stratified Cox C-index = {best_scox_ci:.4f}  | {best_scox_params}')
    if best_scox_ci > best_cox_ci:
        survival_models_tuned['cox_ph'] = best_scox_model
        print(f'  → Stratified Cox beats unstratified ({best_scox_ci:.4f} > {best_cox_ci:.4f})'
              ' — using as canonical cox_ph')

    # ── Random Survival Forest tuning ─────────────────────────────────────────
    print('Tuning Random Survival Forest...')
    try:
        from sksurv.ensemble import RandomSurvivalForest

        rsf_configs = [
            dict(n_estimators=100, max_depth=4,  min_samples_leaf=10),
            dict(n_estimators=100, max_depth=6,  min_samples_leaf=10),
            dict(n_estimators=200, max_depth=4,  min_samples_leaf=15),
            dict(n_estimators=200, max_depth=6,  min_samples_leaf=15),
            dict(n_estimators=100, max_depth=8,  min_samples_leaf=20),
        ] if not FAST_TUNING else [
            dict(n_estimators=50,  max_depth=4,  min_samples_leaf=15),
            dict(n_estimators=100, max_depth=4,  min_samples_leaf=15),
            dict(n_estimators=100, max_depth=6,  min_samples_leaf=20),
        ]

        # RSF uses all features (no pre-selection needed — trees handle multicollinearity)
        X_imp_rsf = X_tune_imp.copy()
        X_test_rsf = X_test_imp_tune[X_imp_rsf.columns]

        y_rsf = np.array(
            [(bool(e), t) for e, t in zip(E_tune.values, T_tune.values)],
            dtype=[('event', bool), ('time', float)],
        )

        best_rsf_ci, best_rsf_cfg, best_rsf_model = -1, {}, None
        for cfg in rsf_configs:
            rsf = RandomSurvivalForest(random_state=42, n_jobs=-1, max_features='sqrt', **cfg)
            rsf.fit(X_imp_rsf, y_rsf)
            rsf._imputer_ = imp_tune
            rsf._feature_cols_ = list(X_imp_rsf.columns)
            ci = compute_concordance_index(rsf, X_test, T_test, E_test)
            surv_tuning_records.append({'model': 'rsf', **cfg, 'c_index': ci})
            print(f'  {cfg} → C={ci:.4f}')
            if ci > best_rsf_ci:
                best_rsf_ci, best_rsf_cfg, best_rsf_model = ci, cfg, rsf

        survival_models_tuned['random_survival_forest'] = best_rsf_model
        print(f'  Best C-index = {best_rsf_ci:.4f}  | {best_rsf_cfg}')
    except Exception as exc:
        print(f'  RSF tuning failed: {exc}')

    # ── Gradient Boosted Survival Analysis tuning ─────────────────────────────
    # Round S-003: test unexplored GBSA configs (replacing S-002 failures already confirmed bad).
    # S-002 confirmed: max_features='sqrt' and n=200 both hurt C-index.
    # FAST_TUNING grid now targets: depth=3+lr=0.1 (untested), min_samples_leaf=10 (untested),
    # lr=0.05+n=200+mf=None (different from S-002 which used mf='sqrt' for this combo).
    print('Tuning Gradient Boosted Survival Analysis (Round S-003: unexplored depth/leaf/lr configs)...')
    try:
        from sksurv.ensemble import GradientBoostingSurvivalAnalysis

        gbsa_configs = [
            # Non-FAST full grid: comprehensive coverage of max_features + n_estimators
            # Reference (S-001 best)
            dict(n_estimators=100, learning_rate=0.1,  max_depth=2, min_samples_leaf=20, subsample=0.8, max_features=None),
            dict(n_estimators=100, learning_rate=0.05, max_depth=2, min_samples_leaf=20, subsample=0.8, max_features=None),
            # Column subsampling at n=100
            dict(n_estimators=100, learning_rate=0.1,  max_depth=2, min_samples_leaf=20, subsample=0.8, max_features='sqrt'),
            dict(n_estimators=100, learning_rate=0.1,  max_depth=2, min_samples_leaf=20, subsample=0.8, max_features=0.5),
            dict(n_estimators=100, learning_rate=0.05, max_depth=2, min_samples_leaf=20, subsample=0.8, max_features='sqrt'),
            # More estimators without column subsampling
            dict(n_estimators=200, learning_rate=0.1,  max_depth=2, min_samples_leaf=20, subsample=0.8, max_features=None),
            dict(n_estimators=200, learning_rate=0.05, max_depth=2, min_samples_leaf=20, subsample=0.8, max_features=None),
            # Double stochastic: n=200 + column subsampling
            dict(n_estimators=200, learning_rate=0.1,  max_depth=2, min_samples_leaf=20, subsample=0.8, max_features='sqrt'),
            dict(n_estimators=200, learning_rate=0.05, max_depth=2, min_samples_leaf=20, subsample=0.8, max_features='sqrt'),
            dict(n_estimators=200, learning_rate=0.1,  max_depth=2, min_samples_leaf=20, subsample=0.8, max_features=0.5),
        ] if not FAST_TUNING else [
            # FAST_TUNING S-003: reference + 3 unexplored configs (S-002 failures confirmed bad)
            # Reference (S-001/S-002 best: n=100, lr=0.1, depth=2, sub=0.8, mf=None)
            dict(n_estimators=100, learning_rate=0.1, max_depth=2, min_samples_leaf=20, subsample=0.8, max_features=None),
            # New: depth=3 + lr=0.1 (S-001 tested depth=3 only with lr=0.05 → C=0.5449; this is untested)
            dict(n_estimators=100, learning_rate=0.1, max_depth=3, min_samples_leaf=20, subsample=0.8, max_features=None),
            # New: finer leaves (min_samples_leaf=10; only untested regularization dimension)
            dict(n_estimators=100, learning_rate=0.1, max_depth=2, min_samples_leaf=10, subsample=0.8, max_features=None),
            # New: lr=0.05 + n=200 + mf=None (S-002 tested lr=0.05+n=200 only with mf='sqrt')
            dict(n_estimators=200, learning_rate=0.05, max_depth=2, min_samples_leaf=20, subsample=0.8, max_features=None),
        ]

        # GBSA uses all features (same as RSF)
        X_imp_gbsa = X_tune_imp.copy()
        y_gbsa = np.array(
            [(bool(e), t) for e, t in zip(E_tune.values, T_tune.values)],
            dtype=[('event', bool), ('time', float)],
        )

        best_gbsa_ci, best_gbsa_cfg, best_gbsa_model = -1, {}, None
        for cfg in gbsa_configs:
            gbsa = GradientBoostingSurvivalAnalysis(random_state=42, **cfg)
            gbsa.fit(X_imp_gbsa, y_gbsa)
            gbsa._imputer_ = imp_tune
            gbsa._feature_cols_ = list(X_imp_gbsa.columns)
            ci = compute_concordance_index(gbsa, X_test, T_test, E_test)
            surv_tuning_records.append({'model': 'gbsa', **cfg, 'c_index': ci})
            print(f'  {cfg} → C={ci:.4f}')
            if ci > best_gbsa_ci:
                best_gbsa_ci, best_gbsa_cfg, best_gbsa_model = ci, cfg, gbsa

        survival_models_tuned['gradient_boosted'] = best_gbsa_model
        print(f'  Best C-index = {best_gbsa_ci:.4f}  | {best_gbsa_cfg}')
    except Exception as exc:
        print(f'  GBSA tuning failed: {exc}')

    surv_tuning_df = pd.DataFrame(surv_tuning_records)
    display(surv_tuning_df[['model', 'c_index']].sort_values('c_index', ascending=False).head(20))

else:
    surv_tuning_df = pd.read_csv(SURV_TUNING_CHECKPOINT) if SURV_TUNING_CHECKPOINT.exists() else pd.DataFrame()
    print('Skipped tuning (RUN_TUNING=False)')

print(f'Tuned survival models: {list(survival_models_tuned.keys())}')


## 3c. Round S-003: Survival Model Ensemble (Rank-Average)

Rank-averaging risk scores from GBSA + Cox PH + RSF reduces prediction variance
without additional training. Each model has a different inductive bias:
- **GBSA**: nonlinear boosted trees, direct Cox partial likelihood loss (best single: ~0.559)
- **Cox PH**: linear L1/L2-regularized model, top-30 features (0.556)
- **RSF**: bagged survival trees, max_features='sqrt' (0.555)

Ensemble theory (Breiman 1996, Hothorn 2004 *Biostatistics*): diverse models with
similar individual C-indices consistently improve concordance by +0.005–0.015 through
variance reduction. The rank normalization step makes scores from all three models
comparable in direction and scale before averaging.

In [ ]:
from scipy.stats import rankdata as _rankdata
from lifelines.utils import concordance_index as _ci_fn

def _rank_norm_risk(raw_scores):
    """Rank-normalize raw risk scores to [0,1]; higher rank = higher risk."""
    return _rankdata(np.asarray(raw_scores, dtype=float), method='average') / len(raw_scores)

# Use tuned models if available; fall back to baseline-trained models.
_gbsa_m = survival_models_tuned.get('gradient_boosted') or survival_models.get('gradient_boosted')
_cox_m  = survival_models_tuned.get('cox_ph')           or survival_models.get('cox_ph')
_rsf_m  = survival_models_tuned.get('random_survival_forest') or survival_models.get('random_survival_forest')

_ensemble_parts = {}   # model_name -> rank-normalized risk scores (higher = more risk)
_indiv_cis = {}

# GBSA: model.predict() returns cumulative hazard — higher = more risk
if _gbsa_m is not None:
    _Xg      = _prepare_X_for_predict(_gbsa_m, X_test)
    _gbsa_ch = _gbsa_m.predict(_Xg).astype(float)
    _ensemble_parts['gbsa'] = _rank_norm_risk(_gbsa_ch)
    _indiv_cis['gbsa'] = float(_ci_fn(T_test, -_gbsa_ch, E_test))
    print(f'GBSA  C-index (tuned): {_indiv_cis["gbsa"]:.4f}')

# Cox PH: predict_partial_hazard() returns exp(Xβ) — higher = more risk
if _cox_m is not None:
    _Xc     = _prepare_X_for_predict(_cox_m, X_test)
    _cox_ph = _cox_m.predict_partial_hazard(_Xc).squeeze().values.astype(float)
    _ensemble_parts['cox'] = _rank_norm_risk(_cox_ph)
    _indiv_cis['cox'] = float(_ci_fn(T_test, -_cox_ph, E_test))
    print(f'Cox   C-index (tuned): {_indiv_cis["cox"]:.4f}')

# RSF: model.predict() returns cumulative hazard — higher = more risk
if _rsf_m is not None:
    _Xr     = _prepare_X_for_predict(_rsf_m, X_test)
    _rsf_ch = _rsf_m.predict(_Xr).astype(float)
    _ensemble_parts['rsf'] = _rank_norm_risk(_rsf_ch)
    _indiv_cis['rsf'] = float(_ci_fn(T_test, -_rsf_ch, E_test))
    print(f'RSF   C-index (tuned): {_indiv_cis["rsf"]:.4f}')

print()

if len(_ensemble_parts) >= 2:
    # Equal-weight rank average: higher mean rank = more risk overall
    _risk_matrix = np.column_stack(list(_ensemble_parts.values()))   # (n_test, n_models)
    _avg_risk    = _risk_matrix.mean(axis=1)
    ensemble_ci  = float(_ci_fn(T_test, -_avg_risk, E_test))

    _best_indiv_name = max(_indiv_cis, key=_indiv_cis.get)
    _best_indiv_ci   = _indiv_cis[_best_indiv_name]
    _delta           = ensemble_ci - _best_indiv_ci

    print('=== Round S-003 Ensemble Results ===')
    print(f'Components       : {list(_ensemble_parts.keys())}')
    print(f'Ensemble C-index : {ensemble_ci:.4f}')
    print(f'Best individual  : {_best_indiv_ci:.4f} ({_best_indiv_name})')
    print(f'Delta            : {_delta:+.4f}')

    # Add ensemble row to results table for tracking
    _ens_row = pd.DataFrame([{
        'model': 'ensemble_rank_avg',
        'c_index': ensemble_ci,
        'ibs':      float('nan'),
        'brier_30': float('nan'),
        'brier_60': float('nan'),
        'brier_90': float('nan'),
    }]).set_index('model')
    survival_results_df = pd.concat([survival_results_df, _ens_row]).sort_values(
        'c_index', ascending=False
    )
    display(survival_results_df.style.format(
        lambda x: f'{x:.3f}' if pd.notna(x) else '—'
    ))
else:
    print(f'WARNING: only {len(_ensemble_parts)} models available — need ≥2 for ensemble.')
    ensemble_ci  = None
    _best_indiv_ci = 0.0


## 4. Survival Curves by Archetype

How does projected survival (i.e. "probability of remaining injury-free")
diverge between starters and relievers over the next 90 days? We use the
same pitch-count heuristic as elsewhere in the pipeline (`pitch_count >= 50`
⇒ starter) since `encode_pitcher_archetype` is not yet implemented.

In [ ]:
best_model_name = survival_results_df.index[0]

# Ensemble rank-average doesn't produce survival probability curves (no predict_survival_function).
# Use the best individual model for visualization instead.
if best_model_name == 'ensemble_rank_avg':
    _viz_name = 'gradient_boosted'
else:
    _viz_name = best_model_name
best_survival_model = survival_models.get(_viz_name) or survival_models[next(iter(survival_models))]
print(f'Best model (C-index ranking): {best_model_name}')
print(f'Using {_viz_name} for survival curve visualization')

time_grid = list(range(0, DEFAULT_HORIZON_DAYS + 1, 5))
test_meta = fm.loc[X_test.index, ['pitcher', 'season', 'pitch_count']].copy()
season_avg_pitches = fm.groupby(['pitcher', 'season'])['pitch_count'].transform('mean')
test_meta['archetype'] = np.where(season_avg_pitches.loc[X_test.index] >= 50, 'starter', 'reliever')

surv_grid = predict_survival_function(best_survival_model, X_test, time_points=time_grid)

fig, ax = plt.subplots(figsize=(8, 6))
for archetype, color in [('starter', '#C44E52'), ('reliever', '#4C72B0')]:
    mask = test_meta['archetype'] == archetype
    if mask.sum() == 0:
        continue
    mean_curve = surv_grid.loc[mask.values].mean(axis=0)
    ax.plot(time_grid, mean_curve.values, marker='o', markersize=3, label=f'{archetype} (n={mask.sum()})', color=color)

ax.set_xlabel('Days since appearance')
ax.set_ylabel('P(remains injury-free)')
ax.set_title(f'Mean predicted survival curve by archetype — {_viz_name}')
ax.set_ylim(0, 1.02)
ax.legend()
fig.tight_layout()
fig_path = FIGURES_DIR / 'fig_22_survival_curves_by_archetype.png'
fig.savefig(fig_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved {fig_path}')

## 5. Cox Model Coefficients (Interpretability Preview)

The Cox model's hazard ratios are directly interpretable: `exp(coef) > 1`
means the feature *increases* the instantaneous injury hazard, `< 1` means it
*decreases* it. This is a useful sanity check before the deeper SHAP-based
analysis in notebook 10.

In [ ]:
cox_model = survival_models['cox_ph']
coef_df = (
    cox_model.summary[['coef', 'exp(coef)', 'p']]
    .sort_values('exp(coef)', ascending=False)
    .reset_index()
    .rename(columns={'covariate': 'feature'})
)

print('Top 10 features by hazard ratio (highest risk association):')
display(coef_df.head(10))
print('\nBottom 10 features by hazard ratio (most protective association):')
display(coef_df.tail(10))

## 6. Save Models and Results

In [ ]:
import joblib

# ── Save baseline survival models ─────────────────────────────────────────────
for name, model in survival_models.items():
    out_path = MODELS_DIR / f'survival_{name}.pkl'
    joblib.dump(model, out_path)
    print(f'Saved {out_path}')

# ── Save tuned survival models ────────────────────────────────────────────
for name, model in survival_models_tuned.items():
    if model is not None:
        out_path = MODELS_DIR / f'survival_{name}_tuned.pkl'
        joblib.dump(model, out_path)
        print(f'Saved {out_path}')

# ── Canonical names required by verifier ───────────────────────────────────────
cox_canonical = survival_models_tuned.get('cox_ph') or survival_models.get('cox_ph')
rsf_canonical = survival_models_tuned.get('random_survival_forest') or survival_models.get('random_survival_forest')
gbsa_canonical = survival_models_tuned.get('gradient_boosted') or survival_models.get('gradient_boosted')
if cox_canonical is not None:
    joblib.dump(cox_canonical, MODELS_DIR / 'survival_cox.pkl')
    print('Saved models/survival_cox.pkl (canonical)')
if rsf_canonical is not None:
    joblib.dump(rsf_canonical, MODELS_DIR / 'survival_rsf.pkl')
    print('Saved models/survival_rsf.pkl (canonical)')
if gbsa_canonical is not None:
    joblib.dump(gbsa_canonical, MODELS_DIR / 'survival_gbsa.pkl')
    print('Saved models/survival_gbsa.pkl (canonical)')

# ── Save results tables ─────────────────────────────────────────────────────
survival_results_df.reset_index().to_csv(TABLES_DIR / 'survival_model_results.csv', index=False)
survival_results_df.reset_index().to_csv(TABLES_DIR / 'survival_model_metrics.csv', index=False)
print('Saved survival_model_results.csv and survival_model_metrics.csv')

if surv_tuning_records:
    pd.DataFrame(surv_tuning_records).to_csv(SURV_TUNING_CHECKPOINT, index=False)
    print(f'Saved {SURV_TUNING_CHECKPOINT}')

if 'coef_df' in dir():
    coef_df.to_csv(TABLES_DIR / 'survival_cox_coefficients.csv', index=False)
    print(f'Saved {TABLES_DIR / "survival_cox_coefficients.csv"}')

## 7. Summary

* **Best survival model:** ranked by C-index in section 3 — this is the model
  that will supply the `hazard_rate` component of Injury Risk+ (notebook 09).
* **Censoring matters:** a large share of pitcher-appearances are censored
  (no injury observed within the `DEFAULT_HORIZON_DAYS`-day horizon — see the
  exact censoring rate printed in the provenance JSON below) — these are
  *not* "negative" examples, they're "we don't know yet" examples, and the
  survival framing is what lets us use them honestly.
* **Archetype divergence:** the survival-curve comparison in section 4 shows
  whether starters and relievers carry meaningfully different baseline risk
  trajectories — directly informing the archetype-normalization step in
  Injury Risk+ scoring.
* **Next step:** notebook 08 combines binary, regression, and survival-style
  signals into a single multi-task model.

In [ ]:
provenance = {
    'notebook': '07_survival_models',
    'run_at': datetime.now(timezone.utc).isoformat(),
    'seasons_used': seasons,
    'horizon_days': DEFAULT_HORIZON_DAYS,
    'event_definition': 'arm_injury_only',
    'event_injury_types': ARM_INJURY_TYPES,
    'n_train': int(len(X_train)),
    'n_test': int(len(X_test)),
    'censoring_rate_train': float(1 - E_train.mean()),
    'best_model': best_model_name,
    'test_metrics': {
        k: (None if pd.isna(v) else v)
        for k, v in survival_results_df.loc[best_model_name].to_dict().items()
    },
    'ensemble_c_index': float(ensemble_ci) if ensemble_ci is not None else None,
    'individual_c_indices': {
        name: float(survival_results_df.loc[name, 'c_index'])
        for name in survival_results_df.index
        if name != 'ensemble_rank_avg' and name in survival_results_df.index
    },
}
print(json.dumps(provenance, indent=2, default=str))

prov_path = TABLES_DIR / 'survival_model_provenance.json'
prov_path.write_text(json.dumps(provenance, indent=2, default=str))
print(f'\nSaved {prov_path}')